# Sheet 2.3: Modern LLM architectures (Llama, Qwen3, DeepSeek)

**Roadmap topic 2.3** (Fanout 08.01-08.03, 12.02). Read first: Raschka's *Big LLM Architecture Comparison*, the RoFormer (RoPE), GLU-variants and GQA papers, the Qwen3 technical report, DeepSeek-V3 sections on MLA and MoE. Study `rasbt/LLMs-from-scratch/ch05/11_qwen3`.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. You may use `F.scaled_dot_product_attention` from here on. Checks run on CPU with randomly initialised Hugging Face models (no downloads).

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

## Exercise 1: Rotary position embeddings

Use the Hugging Face / Llama convention ("rotate half"): split the head dimension into two halves `x1, x2` and rotate pairs `(x1[i], x2[i])` by angle `pos · base^(-2i/d)`.
- `rope_cos_sin(T, head_dim, base)` returns `cos, sin` of shape `(T, head_dim)`.
- `apply_rope(x, cos, sin)` for `x (B, H, T, head_dim)`.

The check verifies the property that makes RoPE work: after rotation, `q_m · k_n` depends only on `m - n`.

In [ ]:
def rope_cos_sin(T, head_dim, base=10000.0, dtype=torch.float32):
    # YOUR CODE
    raise NotImplementedError
def apply_rope(x, cos, sin):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    d = 64; cos, sin = rope_cos_sin(200, d, dtype=torch.float64)
    assert cos.shape == (200, d)
    q = torch.randn(1, 1, 1, d, dtype=torch.float64); k = torch.randn(1, 1, 1, d, dtype=torch.float64)
    def dot(m, n): return float((apply_rope(q, cos[m], sin[m]) * apply_rope(k, cos[n], sin[n])).sum())
    for m, n, s in ((5, 2, 40), (100, 7, 90), (3, 3, 150)):
        assert abs(dot(m, n) - dot(m + s, n + s)) < 1e-9, "q.k must depend only on m - n"
    x = torch.randn(2, 3, 10, d, dtype=torch.float64)
    assert torch.allclose(apply_rope(x, cos[:10], sin[:10]).norm(dim=-1), x.norm(dim=-1)), "rotation must preserve norm"
    assert torch.allclose(apply_rope(x[:, :, :1], cos[:1], sin[:1]), x[:, :, :1]), "position 0 must be the identity"
    print("PASS")
_check1()

**Written (1a):** Why does RoPE use a geometric range of frequencies? What happens to the lowest frequencies at long context, and what do "RoPE scaling" methods (NTK-aware, YaRN) change?

**Your answer:**

## Exercise 2: A Qwen3 model from scratch that matches Hugging Face

Implement `Qwen3(cfg)` and `load_hf(model, state_dict)`. Qwen3 = Llama-style decoder plus **QK-norm**:
- token embedding (tied LM head when `cfg.tie_word_embeddings`)
- each layer: `x + Attn(RMSNorm(x))`, then `x + MLP(RMSNorm(x))`
- attention: no biases, `num_attention_heads` query heads and `num_key_value_heads` KV heads (**GQA**: repeat each KV head for its group), `head_dim` may differ from `hidden/heads`, **RMSNorm on q and k per head (over `head_dim`) before RoPE**, causal
- MLP: SwiGLU, `down(silu(gate(x)) * up(x))`
- final RMSNorm, eps from `cfg.rms_norm_eps`, RoPE base from `cfg.rope_parameters["rope_theta"]`

Use your own RMSNorm (weight · x / rms) computed in float32.

In [ ]:
class Qwen3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # YOUR CODE
        raise NotImplementedError
    def forward(self, idx):          # (B, T) -> logits (B, T, vocab)
        # YOUR CODE
        raise NotImplementedError

def load_hf(model, sd):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
@torch.no_grad()
def _check2():
    from transformers import Qwen3Config, Qwen3ForCausalLM
    torch.manual_seed(0)
    cfg = Qwen3Config(vocab_size=500, hidden_size=128, intermediate_size=344, num_hidden_layers=3, num_attention_heads=8,
                      num_key_value_heads=2, head_dim=32, max_position_embeddings=256, tie_word_embeddings=True,
                      bos_token_id=0, eos_token_id=0, pad_token_id=0, attn_implementation="eager")
    hf = Qwen3ForCausalLM(cfg).eval()
    for n, p in hf.named_parameters():
        if "norm" in n: p.copy_(1 + 0.1 * torch.randn_like(p))          # make the norms non-trivial
    mine = Qwen3(cfg).eval(); load_hf(mine, hf.state_dict())
    idx = torch.randint(0, 500, (2, 50))
    a, b = mine(idx), hf(idx).logits
    assert torch.allclose(a, b, atol=1e-4), f"logits differ, max {(a - b).abs().max():.2e}"
    print(f"PASS  logits match HF Qwen3 (max diff {(a - b).abs().max():.1e})")
_check2()

### On Colab: the real thing
Load `Qwen/Qwen3-0.6B` weights into your class (`AutoModelForCausalLM.from_pretrained(..., torch_dtype=torch.float32)`), confirm logits match within 1e-3 on 20 prompts, then greedy-decode 50 tokens from your model.

Then do the ablation study from the dashboard: start from your GPT-2 (sheet 2.2) and switch on one change at a time (RMSNorm, RoPE, SwiGLU, GQA, no biases, QK-norm) on TinyStories at a fixed token budget, 2 seeds each. One plot, one table.

In [ ]:
# Colab: Qwen3-0.6B parity + architecture ablations

## Exercise 3: Mixture of experts with load balancing

`TopKMoE(d, d_ff, n_experts, k)`: each expert is a SwiGLU MLP. The router is a linear layer → softmax over experts → keep the top-`k` probabilities and **renormalise them to sum to 1** (Mixtral style). `forward(x)` takes `(N, d)` tokens and returns `(y, aux_loss)`.

Auxiliary load-balancing loss (Switch Transformer, generalised to top-k): `aux = n_experts · Σᵢ fᵢ · Pᵢ`, where `fᵢ` is the fraction of the `N·k` token-to-expert assignments that went to expert `i`, and `Pᵢ` is the mean router probability (before top-k) for expert `i`.

Only run each expert on the tokens routed to it (gather/scatter with `index_add_`), not on all tokens.

In [ ]:
class TopKMoE(nn.Module):
    def __init__(self, d, d_ff, n_experts, k):
        super().__init__()
        # YOUR CODE
        raise NotImplementedError
    def forward(self, x):
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    torch.manual_seed(0); d, E = 32, 6
    m = TopKMoE(d, 64, E, E); x = torch.randn(40, d)
    y, aux = m(x)
    p = torch.softmax(m.router(x), -1)
    dense = sum(p[:, e:e + 1] * m.experts[e](x) for e in range(E))
    assert torch.allclose(y, dense, atol=1e-5), "with k = n_experts the output must equal the dense mixture"
    m2 = TopKMoE(d, 64, E, 2); y2, aux2 = m2(x)
    p2 = torch.softmax(m2.router(x), -1); w, idx = p2.topk(2, -1); w = w / w.sum(-1, keepdim=True)
    ref = torch.stack([sum(w[i, j] * m2.experts[int(idx[i, j])](x[i:i + 1])[0] for j in range(2)) for i in range(40)])
    assert torch.allclose(y2, ref, atol=1e-5), "top-2 output wrong"
    f = torch.bincount(idx.flatten(), minlength=E).float() / 80
    assert torch.allclose(aux2, E * (f * p2.mean(0)).sum()), "aux loss formula wrong"
    with torch.no_grad(): m2.router.weight.zero_()
    _, aux_u = m2(x)
    assert abs(aux_u.item() - 1.0) < 1e-5, "uniform router must give aux = 1"
    y2.sum().backward(); assert m2.router.weight.grad is not None, "router must receive gradient through the gate weights"
    print("PASS")
_check3()

**Experiment (CPU ok):** train a 2-layer model with an 8-expert, top-2 MoE MLP on a toy task (or TinyStories on Colab) with aux-loss weight 0 and 0.01. Plot the per-expert token counts over training. You should see collapse to a few experts without the aux loss.

## Exercise 4: KV-cache budget

`kv_cache_bytes_per_token(cfg)` where `cfg` is a dict with `n_layers`, `bytes_per_el`, and `kind` in `{"mha", "gqa", "mla"}`:
- `mha`: needs `n_heads`, `head_dim`
- `gqa`: needs `n_kv_heads`, `head_dim`
- `mla`: needs `kv_lora_rank`, `rope_head_dim` (DeepSeek caches the compressed latent plus the decoupled RoPE key, shared across heads)

In [ ]:
def kv_cache_bytes_per_token(cfg):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    llama8b_mha = dict(kind="mha", n_layers=32, n_heads=32, head_dim=128, bytes_per_el=2)
    llama8b = dict(kind="gqa", n_layers=32, n_kv_heads=8, head_dim=128, bytes_per_el=2)
    dsv3 = dict(kind="mla", n_layers=61, kv_lora_rank=512, rope_head_dim=64, bytes_per_el=2)
    dsv3_if_mha = dict(kind="mha", n_layers=61, n_heads=128, head_dim=128 + 64, bytes_per_el=2)
    a, b, c, d = map(kv_cache_bytes_per_token, (llama8b_mha, llama8b, dsv3, dsv3_if_mha))
    assert (a, b, c) == (524288, 131072, 70272), (a, b, c)
    print(f"PASS  per token: Llama-3-8B GQA {b/1024:.0f} KiB (MHA would be {a/1024:.0f}); DeepSeek-V3 MLA {c/1024:.1f} KiB "
          f"vs {d/2**20:.1f} MiB with plain MHA. At 128k context: Llama-3-8B {b*131072/2**30:.0f} GiB, DeepSeek-V3 {c*131072/2**30:.1f} GiB")
_check4()

**Written (4a):** MLA cuts the cache by roughly 80× vs plain MHA at DeepSeek's size. What does it cost in compute at decode time, and what trick ("absorbing" the up-projection) removes most of that cost?

## Checkpoint
Pick any model's `config.json` on the Hugging Face Hub (e.g. Llama-3.2-1B, Qwen3-4B, Mistral-7B). By hand, sketch the forward pass and compute the parameter count and KV-cache bytes per token. Compare with the model card; you should be within 5%.

---
### Hints for self-grading
- 1a: different frequencies give each pair of dimensions a different "wavelength", so attention can be sensitive to both local and distant offsets. At contexts longer than training, low-frequency dimensions see rotation angles they never saw; NTK/YaRN rescale the base or interpolate frequencies so angles stay in the trained range.
- 4a: the latent must be up-projected to per-head keys/values (extra matmuls). Because the up-projection is linear, it can be folded into the query and output projections, so attention runs directly on the latent.